# Zeitansage_Agent: ein Agent mit einem Tool

Ein **Agent** ist ein Sprachmodell (hier Gemini), das selbst entscheidet, ob es ein
**Tool** braucht. Ein Tool ist eine normale Python-Funktion, die der Agent aufrufen darf.
Unser Agent hat ein Tool: Es sagt die Uhrzeit in der Schweiz und wie lange der Kurstag noch dauert.

## Vorbereitung (einmal)
1. **Kopie speichern:** *Datei → Kopie in Drive speichern*. Sonst gehen deine Änderungen verloren.
2. **API-Key hinterlegen:** links auf das **Schlüssel-Symbol** (Secrets) klicken,
   *Neues Secret hinzufügen*, Name `API_KEY`, als Wert den Key von Moodle einfügen,
   **Notebook-Zugriff** einschalten. Der Key steht so nie im Notebook selbst.

## Jedes Mal
Colab löscht nach einer Pause alles. Führe die Zellen darum **von oben nach unten** aus
(Play-Knopf links an jeder Zelle, oder *Laufzeit → Alle ausführen*).

> **Tipp für den Code-Assistenten Gemini:** Schreib in deine Anfrage dazu:
> *«Nutze Google ADK, Modell `gemini-3.5-flash`, halte den Code so einfach wie möglich,
> Kommentare auf Deutsch.»*

## 1. Einrichten

In [ ]:
# Google ADK (Agent Development Kit) installieren. Dauert ca. 1 Minute.
!pip install -q "google-adk>=2.15,<3"

In [ ]:
# API-Key aus den Colab-Secrets holen (Schlüssel-Symbol links).
import os
from google.colab import userdata

# ADK sucht den Key unter dem Namen GOOGLE_API_KEY.
os.environ["GOOGLE_API_KEY"] = userdata.get("API_KEY")
print("Key geladen")

## 2. Das Tool

Wichtig ist der **Docstring** (der Text in `""" """`): Daran erkennt Gemini, wofür das Tool gut ist.
`tool_context` gibt ADK automatisch mit. Darüber kann das Tool den Ablauf beeinflussen (siehe Demo unten).

In [ ]:
import math
from datetime import datetime
from zoneinfo import ZoneInfo

from google.adk.tools import ToolContext


def aktuelle_uhrzeit(tool_context: ToolContext) -> dict:
    """Gibt die aktuelle Uhrzeit in der Schweiz zurück, mit einem Hinweis zum Kurstag."""
    jetzt = datetime.now(ZoneInfo("Europe/Zurich"))  # Colab läuft in UTC, darum die Zeitzone
    kursende = jetzt.replace(hour=17, minute=15, second=0, microsecond=0)

    if jetzt < kursende:
        rest_minuten = math.ceil((kursende - jetzt).total_seconds() / 60)
        stunden = rest_minuten // 60
        minuten = rest_minuten % 60
        hinweis = f"CAS Kurstag: Dann musst du noch maximal {stunden} Stunden und {minuten} Minuten durchhalten."
    else:
        hinweis = "Feierabend!"

    ansage = f"Es ist {jetzt:%H:%M} Uhr. {hinweis}"

    # tool_context.actions.skip_summarization = True

    # --- DEMO: Entferne das # vor der Zeile oben, dann wird das Tool-Ergebnis
    # NICHT an Gemini geschickt.
    # Normalerweise geht das Tool-Ergebnis zurück an Gemini, und Gemini
    # formuliert daraus die Antwort (zweite Modellrunde).
    # Ohne # zeigt ADK das Ergebnis direkt an, und Gemini sieht es nie.
    # -------------------------------------------------------------------------

    return {"status": "ok", "ansage": ansage}

## 3. Der Agent

- `model`: welches Sprachmodell denkt
- `description`: kurze Beschreibung, wichtig, wenn später mehrere Agenten zusammenarbeiten
- `instruction`: die Anweisung an das Modell (der «Systemprompt»)
- `tools`: die Funktionen, die der Agent aufrufen darf

In [ ]:
from google.adk.agents import Agent

root_agent = Agent(
    name="Zeitansage_Agent",
    model="gemini-3.5-flash",
    description="Sagt die aktuelle Uhrzeit an und wie lange der Kurstag noch dauert.",
    instruction=(
        "Du bist ein freundlicher Assistent in einem Weiterbildungskurs. "
        "Antworte kurz und auf Deutsch. "
        "Wenn nach Datum, Uhrzeit oder Kursende gefragt wird, rufe das Tool "
        "aktuelle_uhrzeit auf und gib dessen Feld 'ansage' mit einer kleinen total lustigen Variation wieder."
    ),
    tools=[aktuelle_uhrzeit],
)

## 4. Den Agenten fragen

Der **Runner** führt den Agenten aus und merkt sich das Gespräch.
`verbose=True` zeigt auch, welches Tool aufgerufen wurde und was es zurückgab.
Ändere die Frage und führe die Zelle nochmal aus.

In [ ]:
from google.adk.runners import InMemoryRunner

runner = InMemoryRunner(agent=root_agent)
await runner.run_debug("Wie spät ist es?", verbose=True)

## 5. Ausprobieren

1. **Eine Frage ohne Tool:** Frag zum Beispiel *«Erzähl mir einen Witz.»* Ruft der Agent das Tool auf?
2. **Ein Gespräch:** Der Runner merkt sich, was vorher gesagt wurde, bis du die Zelle oben
   wieder ausführst. Führe die Zelle unten mehrmals mit verschiedenen Fragen aus.
3. **Demo `skip_summarization`:** Entferne im Tool das `#` vor `tool_context.actions...`.
   Führe danach die Tool-Zelle, die Agent-Zelle und die Frage-Zelle **nochmal aus**.
   Was ist jetzt anders an der Antwort?

In [ ]:
await runner.run_debug("Und wann ist Feierabend?", verbose=True)